# APT29 port — multi-host, multi-technique

Drop-in replacement for the atomic PORT cell. Produces the same `alerts` DataFrame (columns: `semantic_alert`, `ground_truth_technique`, plus `host`/`process_guid`) so SETUP + CELL A-G run unchanged.

**Key changes from atomic:** streaming load (196k events), host+GUID grouping, multi-technique alerts, and an **indicator-based ground-truth labeler** derived from the published ATT&CK Evals emulation plan.

**At N=424, CELL D auto-switches to `EXTRACT_UNIT='community'`** — clustering now does real work. Also: REMOVE the singleton fallback note in CELL A's reasoning; it won't trigger here.

**Thesis reporting:** report PER-CLASS results, not just overall accuracy (classes are imbalanced — T1112=92 vs T1059.001=1). State the labeler is heuristic/indicator-based and include its rules in an appendix.

In [1]:
# =====================================================================
#  APT29 PORT — multi-host, multi-technique  (Security-Datasets ATT&CK Evals)
#  Scales the atomic prototype to the real benchmark.
#
#  Differences vs the atomic port:
#    1. STREAMING load  — 196k events / 385 MB, so we read line-by-line and
#       keep only Sysmon event types we use (don't build a 385 MB DataFrame).
#    2. host+GUID key   — 4 hosts, so a process is (Hostname, ProcessGuid),
#       not GUID alone (GUIDs can repeat across hosts).
#    3. richer alerts    — adds command-line / powershell / rundll32 / sdelete
#       behaviors that the atomic LSASS-only builder didn't cover.
#    4. REAL ground truth — APT29 has NO per-event technique labels. We derive
#       them transparently from observable indicators (process + command line
#       + access pattern), following the published ATT&CK Evals emulation plan.
#       This labeler is auditable and reported as a method, not hidden.
# =====================================================================
import json, os, urllib.request, zipfile
from collections import defaultdict
import pandas as pd

DAY1_URL = ("https://raw.githubusercontent.com/OTRF/detection-hackathon-apt29/"
            "master/datasets/day1/apt29_evals_day1_manual.zip")

# Sysmon event types we consume (ignore the rest to stay light)
KEEP_EIDS = {1, 3, 7, 8, 10, 11, 12, 13}

# ---------------------------------------------------------------------
# 1. STREAMING LOAD  -> list of slim event dicts (not a giant DataFrame)
# ---------------------------------------------------------------------
def load_apt29(url=DAY1_URL, local_dir="data/apt29", max_events=None):
    os.makedirs(local_dir, exist_ok=True)
    zip_path = os.path.join(local_dir, "apt29_day1.zip")
    if not os.path.exists(zip_path):
        print("Downloading APT29 Day 1 (~14 MB)...")
        urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        name = [n for n in zf.namelist() if n.endswith(".json")][0]
        zf.extractall(local_dir)
    json_path = os.path.join(local_dir, name)

    KEEP_FIELDS = ('EventID', 'Hostname', 'Image', 'CommandLine', 'ParentImage',
                   'ParentCommandLine', 'ProcessGuid', 'User', 'IntegrityLevel',
                   'SourceImage', 'SourceProcessGUID', 'TargetImage',
                   'GrantedAccess', 'ImageLoaded', 'TargetObject', 'EventType',
                   'TargetFilename', 'DestinationIp', 'DestinationPort')
    events = []
    with open(json_path) as f:
        for i, line in enumerate(f):
            if max_events and i >= max_events:
                break
            e = json.loads(line)
            if e.get('EventID') not in KEEP_EIDS:
                continue
            events.append({k: e.get(k) for k in KEEP_FIELDS if k in e})
    print(f"Loaded {len(events)} Sysmon events (filtered to {sorted(KEEP_EIDS)})")
    return events

# ---------------------------------------------------------------------
# 2. GROUP BY (host, process guid)
# ---------------------------------------------------------------------
def _clean(v):
    if v is None or isinstance(v, float):
        return None
    s = str(v).strip()
    return s if s and s.lower() != "nan" else None

def _key(ev):
    host = _clean(ev.get('Hostname')) or 'unknown_host'
    guid = _clean(ev.get('ProcessGuid')) or _clean(ev.get('SourceProcessGUID'))
    return (host, guid) if guid else None

def group_by_process(events):
    procs = defaultdict(lambda: {
        "host": None, "image": None, "command_line": None,
        "parent_image": None, "parent_command_line": None,
        "user": None, "integrity": None,
        "loaded_dlls": [], "accessed": [], "registry": [],
        "files_created": [], "network": [],
    })
    for ev in events:
        if ev.get('EventID') == 1:
            k = _key(ev)
            if not k:
                continue
            p = procs[k]
            p["host"]                = k[0]
            p["image"]               = _clean(ev.get('Image'))
            p["command_line"]        = _clean(ev.get('CommandLine'))
            p["parent_image"]        = _clean(ev.get('ParentImage'))
            p["parent_command_line"] = _clean(ev.get('ParentCommandLine'))
            p["user"]                = _clean(ev.get('User'))
            p["integrity"]           = _clean(ev.get('IntegrityLevel'))
    for ev in events:
        eid = ev.get('EventID')
        k = _key(ev)
        if not k:
            continue
        p = procs[k]
        if p["host"] is None:
            p["host"] = k[0]
        if p["image"] is None:
            p["image"] = _clean(ev.get('Image')) or _clean(ev.get('SourceImage'))
        if eid == 7:
            dll = _clean(ev.get('ImageLoaded'))
            if dll: p["loaded_dlls"].append(dll)
        elif eid == 10:
            p["accessed"].append({"target": ev.get('TargetImage'),
                                  "access": ev.get('GrantedAccess')})
        elif eid in (12, 13, 14):
            p["registry"].append({"key": ev.get('TargetObject'),
                                  "type": ev.get('EventType')})
        elif eid == 11:
            f = _clean(ev.get('TargetFilename'))
            if f: p["files_created"].append(f)
        elif eid in (3, 5156):
            p["network"].append({"dst": ev.get('DestinationIp'),
                                 "port": ev.get('DestinationPort')})
    return procs

# ---------------------------------------------------------------------
# 3. GROUND-TRUTH LABELER  (transparent, indicator-based)
#    Derived from the published APT29 ATT&CK Evals emulation plan. Each rule
#    keys on an OBSERVABLE indicator. Processes with no adversary indicator
#    are labeled 'benign' (not a technique) — this is what lets macro-accuracy
#    be meaningful, unlike the atomic dataset's blanket label.
# ---------------------------------------------------------------------
def label_technique(proc):
    img = (proc["image"] or "").lower()
    cmd = (proc["command_line"] or "").lower()
    accessed_lsass = any("lsass" in str(a["target"]).lower() for a in proc["accessed"])

    # T1003.001 — LSASS credential access (Invoke-Mimikatz step)
    if accessed_lsass and ("powershell" in img or "rundll32" in img
                           or "mimikatz" in cmd):
        return "T1003.001"
    # T1218.011 — rundll32 proxy execution (davclnt.dll DavSetCookie)
    if "rundll32.exe" in img and ("davclnt" in cmd or "davsetcookie" in cmd):
        return "T1218.011"
    # T1059.001 — PowerShell with evasion flags
    if "powershell" in img and any(f in cmd for f in
                                   ("-ep bypass", "-noni", "-window hidden",
                                    "-enc", "bypass", "hidden")):
        return "T1059.001"
    # T1070.004 — file deletion via sdelete (indicator removal)
    if "sdelete" in img:
        return "T1070.004"
    # T1053.005 — scheduled task
    if "schtasks" in img or "schtasks" in cmd:
        return "T1053.005"
    # T1112 — registry modification (only if it's the dominant behavior)
    if proc["registry"] and not proc["command_line"] and not proc["accessed"]:
        return "T1112"
    return "benign"

# ---------------------------------------------------------------------
# 4. SEMANTIC ALERT BUILDER  (atomic builder + command-line behaviors)
# ---------------------------------------------------------------------
def _basename(path):
    return str(path).replace("\\", "/").split("/")[-1] if path else "unknown"

SUSPICIOUS_ACCESS = {"0x1fffff", "0x1010", "0x1410", "0x143a", "0x1438"}

def build_semantic_alert(proc):
    img = _basename(proc["image"])
    sent = []
    if proc["parent_image"]:
        sent.append(f"The process {img} was spawned by {_basename(proc['parent_image'])}.")
    else:
        sent.append(f"Observed activity from the process {img}.")
    if proc["command_line"]:
        sent.append(f"It executed with command line: {proc['command_line'][:200]}.")
    if proc["user"]:
        sent.append(f"Running as {proc['user']} at {proc['integrity'] or 'unknown'} integrity.")

    lsass = [a for a in proc["accessed"] if "lsass" in str(a["target"]).lower()]
    if lsass:
        acc = lsass[0]["access"]
        flag = " with full access rights" if acc in SUSPICIOUS_ACCESS else ""
        sent.append(f"It opened a handle to the LSASS process memory "
                    f"(granted access {acc}){flag}, a behavior associated with "
                    f"credential extraction.")
    elif proc["accessed"]:
        tgts = {_basename(a["target"]) for a in proc["accessed"]}
        sent.append("It accessed the memory of other processes: "
                    f"{', '.join(sorted(tgts)[:4])}.")

    if proc["loaded_dlls"]:
        notable = {_basename(d) for d in proc["loaded_dlls"]
                   if _basename(d).lower() in
                   {"ntdll.dll","dbghelp.dll","dbgcore.dll","samlib.dll",
                    "vaultcli.dll","wlanapi.dll","davclnt.dll"}}
        if notable:
            sent.append("It loaded libraries associated with credential access "
                        f"or proxy execution: {', '.join(sorted(notable))}.")
    if proc["files_created"]:
        sent.append(f"It created {len(proc['files_created'])} file(s) on disk.")
    if proc["registry"]:
        keys = {_basename(r["key"]) for r in proc["registry"] if r["key"]}
        if keys:
            sent.append(f"It modified registry values including "
                        f"{', '.join(list(keys)[:3])}.")
    if proc["network"]:
        sent.append(f"It made {len(proc['network'])} outbound network connection(s).")
    return " ".join(sent)

# ---------------------------------------------------------------------
# 5. BUILD ALERT TABLE
# ---------------------------------------------------------------------
def build_alert_dataframe(events, drop_benign_noise=True):
    procs = group_by_process(events)
    records = []
    for (host, guid), p in procs.items():
        if not (p["accessed"] or p["loaded_dlls"] or p["registry"]
                or p["command_line"] or p["files_created"]):
            continue
        tech = label_technique(p)
        records.append({
            "host": host, "process_guid": guid,
            "image": _basename(p["image"]),
            "semantic_alert": build_semantic_alert(p),
            "ground_truth_technique": tech,
        })
    df = pd.DataFrame(records)
    # keep all malicious; downsample benign so the set isn't 95% noise
    if drop_benign_noise and len(df):
        mal = df[df.ground_truth_technique != "benign"]
        ben = df[df.ground_truth_technique == "benign"].sample(
            n=min(len(mal) * 3, (df.ground_truth_technique == "benign").sum()),
            random_state=42)
        df = pd.concat([mal, ben]).reset_index(drop=True)
    return df

if __name__ == "__main__":
    events = load_apt29()
    alerts = build_alert_dataframe(events)
    print(f"\nBuilt {len(alerts)} process alerts across "
          f"{alerts['host'].nunique()} hosts.")
    print("\nGround-truth technique distribution:")
    print(alerts['ground_truth_technique'].value_counts().to_string())
    print("\nSample malicious alerts:")
    for _, r in alerts[alerts.ground_truth_technique != "benign"].head(5).iterrows():
        print(f"\n[{r['ground_truth_technique']}] {r['image']} on {r['host']}")
        print("  " + r['semantic_alert'][:240])


Loaded 141667 Sysmon events (filtered to [1, 3, 7, 8, 10, 11, 12, 13])

Built 424 process alerts across 4 hosts.

Ground-truth technique distribution:
ground_truth_technique
benign       318
T1112         92
T1070.004      6
T1218.011      5
T1003.001      2
T1059.001      1

Sample malicious alerts:

[T1003.001] powershell.exe on SCRANTON.dmevals.local
  The process powershell.exe was spawned by control.exe. It executed with command line: "PowerShell.exe" -noni -noexit -ep bypass -window hidden -c "sal a New-Object;Add-Type -AssemblyName 'System.Drawing'; $g=a System.Drawing.Bitmap('C:\User

[T1003.001] powershell.exe on SCRANTON.dmevals.local
  The process powershell.exe was spawned by powershell.exe. It executed with command line: powershell.exe. Running as DMEVALS\pbeesly at High integrity. It opened a handle to the LSASS process memory (granted access 0x1000), a behavior associ

[T1070.004] sdelete64.exe on SCRANTON.dmevals.local
  The process sdelete64.exe was spawned by powershe